<a href="https://colab.research.google.com/github/medwicks/BAN482-MachineLearning/blob/main/BAN482_EnsembleMethods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="pink">Adaboost Classifier</font>

### Import Libraries

In [2]:
# Import pandas library
import pandas as pd

# Import numpy library
import numpy as np

# Import DecisionTreeClassifier from sklearn.tree module - this is the base estimator for AdaBoost
from sklearn.tree import DecisionTreeClassifier

# Import GaussianNB (Gaussian Naive Bayes) from sklearn.naive_bayes module - alternative base estimator
from sklearn.naive_bayes import GaussianNB

# Import AdaBoostClassifier from sklearn.ensemble module - the main boosting algorithm we'll use
from sklearn.ensemble import AdaBoostClassifier

### Data Import and Pre-Processing

In [8]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
# Read the flights dataset from CSV file using the correct Google Drive path
flights = pd.read_csv('/content/drive/My Drive/Senior Fall/BAN 482/flights.csv')

# Remove rows with missing values
flights = flights.dropna()

# Display the flights dataframe to verify the data has been loaded successfully
flights.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,year,month,day,dep_time,sched_dep_time,delay,dep_delay,arr_time,sched_arr_time,arr_delay,...,time_hour,temp,dewp,humid,wind_dir,wind_speed,wind_gust,precip,pressure,visib
1,2013,1,1,533.0,529,1,4.0,850.0,830,20.0,...,1/1/13 5:00,39.92,24.98,54.81,250.0,14.96014,21.86482,0.0,1011.4,10.0
4,2013,1,1,554.0,600,0,-6.0,812.0,837,-25.0,...,1/1/13 6:00,39.92,24.98,54.81,260.0,16.11092,23.01560,0.0,1011.7,10.0
7,2013,1,1,557.0,600,0,-3.0,709.0,723,-14.0,...,1/1/13 6:00,39.92,24.98,54.81,260.0,16.11092,23.01560,0.0,1011.7,10.0
9,2013,1,1,558.0,600,0,-2.0,753.0,745,8.0,...,1/1/13 6:00,39.92,24.98,54.81,260.0,16.11092,23.01560,0.0,1011.7,10.0
14,2013,1,1,559.0,600,0,-1.0,941.0,910,31.0,...,1/1/13 6:00,39.92,24.98,54.81,260.0,16.11092,23.01560,0.0,1011.7,10.0


The **flights** dataset contains flight data for all flights that departed all New York City airports (JFK, La Guardia, and Newark) in 2013. Data includes flight destination, scheduled departure time, flight duration, and whether the flight was delayed.

We will create models to predict if a flight will be delayed, using the pre-flight data as input features

In [9]:
# Display the column names
flights.columns

Index(['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'delay',
       'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'carrier',
       'flight', 'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour',
       'minute', 'time_hour', 'temp', 'dewp', 'humid', 'wind_dir',
       'wind_speed', 'wind_gust', 'precip', 'pressure', 'visib'],
      dtype='object')

### Scikit-Learn Workflow

#### DEFINE

In [ ]:
# Create input features X and output feature y

# Create a new dataframe X by selecting specific weather-related columns from the flights dataframe
# These columns will serve as input features (predictors) for the classification model
X = flights[
    [
        'temp',       # Temperature in degrees Fahrenheit - weather condition feature
        'dewp',       # Dew point temperature - measure of atmospheric moisture
        'humid',      # Humidity percentage - amount of water vapor in the air
        'wind_dir',   # Wind direction in degrees - direction from which wind is blowing
        'wind_speed', # Wind speed in mph - velocity of wind
        'wind_gust',  # Wind gust speed in mph - sudden brief increases in wind speed
        'precip',     # Precipitation in inches - amount of rainfall
        'pressure',   # Atmospheric pressure in millibars - weight of air above
        'visib',      # Visibility in miles - how far one can see
    ]
]

# Create target variable y by extracting the 'delay' column from the flights dataframe
# This is the output we want to predict: whether a flight is delayed (1) or not (0)
y = flights[['delay']]

Train-Test Split:
To properly evaluate our models, we should split our dataset into a training set (used to train the model) and a testing set (used to evaluate performance on unseen data). We'll use scikit-learn's `train_test_split` function.

In [ ]:
from sklearn.model_selection import train_test_split

# Split X and y into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=50,
    stratify=y  # Ensure same ratio of delayed to non-delayed flights in both sets
)

# np.ravel(y): converts y from 2D array to 1D array as required by sklearn
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

#### INITIALIZE

In [ ]:
# Initialize the model

# Create an instance of DecisionTreeClassifier with max_depth=3 - this will be the base estimator for AdaBoost
dt_estimator = DecisionTreeClassifier(max_depth=1)

# Create an instance of AdaBoostClassifier using the DecisionTreeClassifier as the base estimator
# AdaBoost (Adaptive Boosting) is an ensemble method that combines multiple weak learners
boostModel = AdaBoostClassifier(estimator=dt_estimator)

#### FIT/TRAIN

In [ ]:

# Fit the model using training data

# Train the AdaBoostClassifier on our data
# X: input features (weather conditions)
# The model learns to predict flight delays by iteratively focusing on misclassified examples

boostModel = boostModel.fit(X_train, y_train)

# Change the max_depth and check effect on accuracy

#### PREDICT

In [ ]:
# Predictions from AdaBoostClassifier()

# Generate predictions for all samples in the input features X
# boostModel.predict(X) returns an array of predicted class labels (0 or 1)
# 0 = no delay, 1 = delay
# This returns the predicted flight delay status for each row in the dataset
y_test_pred = boostModel.predict(X_test)
y_test_pred

#### EVALUATE

In [ ]:
# Generate the transfer matrix
pd.crosstab(
    y_test, # Actual values
    y_test_pred, # Predicted values
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
# Prediction accuracy for AdaBoostClassifier()

# Calculate and display the model's accuracy score on the training data
# boostModel.score(X, y) computes the mean accuracy: the proportion of correct predictions
# X: input features, y: actual target values
# This returns a value between 0 and 1, where 1 means perfect prediction
boostModel.score(X_test, y_test)

In [ ]:
# What was the prediction accurcy on the training data?
boostModel.score(X_train, y_train)

# <font color="pink">Adaboost Regressor</font>

### Import Libraries

In [ ]:
# Import pandas library
import pandas as pd

# Import numpy library
import numpy as np

# Import DecisionTreeRegressor from sklearn.tree module - this will be the base estimator for AdaBoost
from sklearn.tree import DecisionTreeRegressor

# Import AdaBoostRegressor from sklearn.ensemble module - for regression tasks (predicting continuous values)
from sklearn.ensemble import AdaBoostRegressor

# Import LinearRegression from sklearn.linear_model module - this will be the base estimator for AdaBoost
from sklearn.linear_model import LinearRegression

### Data Import and Pre-Processing

In [ ]:
# Read the flights dataset from CSV file, then remove rows with missing values using dropna()
flights = pd.read_csv('flights.csv').dropna()

# Display the flights dataframe to verify the data has been loaded successfully
flights.head()

### Scikit-Learn Workflow

#### DEFINE

In [ ]:
# Create input features X and output feature y
X = flights[
    [
        'dewp',
        'humid',
        'wind_dir',
        'wind_speed',
        'wind_gust',
        'precip',
        'pressure',
        'visib',
    ]
]
y = flights[['temp']]

In [ ]:
from sklearn.model_selection import train_test_split

# Split X and y into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=50
)

# np.ravel(y): converts y from 2D array to 1D array as required by sklearn
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

#### INITIALIZE

In [ ]:
# Initialize the model

# Create an instance of DecisionTreeClassifier with max_depth=3 - this will be the base estimator for AdaBoost
dt_estimator = DecisionTreeRegressor(max_depth=1)

# Create an instance of AdaBoostRegressor using the DecisionTreeRegressor as the base estimator
boostModel = AdaBoostRegressor(estimator=dt_estimator)

#### FIT/TRAIN

In [ ]:
# Fit/Train the model
boostModel = boostModel.fit(X_train, y_train)

#### PREDICT

In [ ]:
# Predictions from AdaBoostRegressor() using the test data
y_test_pred = boostModel.predict(X_test)

#### EVALUATE

In [ ]:
# Compute the Mean Squared Error using test data
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_test_pred)
print(f"Mean Squared Error: {mse}")

# <font color="pink">Bagging Classifier</font>

### Import Libraries

In [ ]:
# Import pandas library
import pandas as pd

# Import numpy library
import numpy as np

# Import BaggingClassifier from sklearn.ensemble module
# Bagging (Bootstrap Aggregating) creates multiple models on different random subsets of the training data
# and combines their predictions through voting to reduce variance and avoid overfitting
from sklearn.ensemble import BaggingClassifier


### Data Import and Pre-Processing

In [ ]:

# Read the CSV file containing flight data and remove any rows with missing values
# pd.read_csv() loads the data from the file
# .dropna() removes all rows that contain any NaN (Not a Number) values
flights = pd.read_csv('flights.csv').dropna()

# Display the flights DataFrame to verify the data has been loaded correctly
# This shows a preview of the data structure, column names, and sample values
flights.head()

### Scikit-Learn Workflow

#### DEFINE

In [ ]:
# Create input features X and output feature y

# Create the feature matrix X by selecting specific weather-related columns from the flights DataFrame
# These columns will serve as the independent variables (predictors) for the machine learning model
X = flights[
       [ 'temp',       # Temperature in degrees Fahrenheit - ambient air temperature
        'dewp',       # Dew point temperature in degrees Fahrenheit - measure of atmospheric moisture
        'humid',      # Humidity percentage - relative humidity in the air
        'wind_dir',   # Wind direction in degrees (0-360) - compass direction from which wind originates
        'wind_speed', # Wind speed in miles per hour - average wind velocity
        'wind_gust',  # Wind gust speed in mph - maximum brief wind speed increases
        'precip',     # Precipitation in inches - amount of rainfall
        'pressure',   # Atmospheric pressure in millibars - barometric pressure measurement
        'visib',      # Visibility in miles - how far one can see clearly
    ]
]

# Create the target variable y by extracting the 'delay' column from the flights DataFrame
# This is the dependent variable (output) that we want to predict: flight delay status (0 or 1)
# 0 indicates no delay, 1 indicates delay
y = flights[['delay']]

In [ ]:
from sklearn.model_selection import train_test_split

# Split X and y into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=50,
    stratify=y  # Ensure same ratio of delayed to non-delayed flights in both sets
)

# np.ravel(y): converts y from 2D array to 1D array as required by sklearn
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

#### INITIALIZE

In [ ]:
# Initialize the model

# Create an instance of BaggingClassifier with default parameters
# BaggingClassifier creates multiple versions of the dataset through bootstrap sampling (random sampling with replacement)
# It trains a base estimator (by default, DecisionTreeClassifier) on each bootstrap sample
# Final prediction is made by majority voting (for classification) from all the base estimators
# This ensemble approach reduces variance and helps prevent overfitting
baggingModel = BaggingClassifier()

#### FIT/TRAIN

In [ ]:
# Fit/Train the model using training data

# Train the BaggingClassifier on the training data
# X: feature matrix containing weather-related input variables
# np.ravel(y): converts the 2D array y into a 1D array, which is required by the fit method
# During training, the model will create multiple bootstrap samples and train separate classifiers on each
# These classifiers will then be combined to make predictions
baggingModel = baggingModel.fit(X_train, y_train)

#### PREDICT

In [ ]:
# Predictions from BaggingClassifier() on the test data
y_test_pred = baggingModel.predict(X_test)

#### EVALUATE

In [ ]:
# Generate the transfer matrix
pd.crosstab(
    y_test, # Actual values
    y_test_pred, # Predicted values
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
# Prediction accuracy for BaggingClassifier()
baggingModel.score(X_test, y_test)

# <font color="pink">Bagging Regressor</font>

### Import Libraries

In [ ]:
# Import packages and data
import pandas as pd
import numpy as np
from sklearn.ensemble import BaggingRegressor

### Data Import and Pre-processing

In [ ]:
flights = pd.read_csv('flights.csv').dropna()

flights.head()

### Scikit-Learn Worfklow

#### DEFINE

In [ ]:
# Create input features X and output feature y
X = flights[
    [
        'dewp',
        'humid',
        'wind_dir',
        'wind_speed',
        'wind_gust',
        'precip',
        'pressure',
        'visib',
    ]
]
y = flights[['temp']]

In [ ]:
from sklearn.model_selection import train_test_split

# Split X and y into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=50
)

# np.ravel(y): converts y from 2D array to 1D array as required by sklearn
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

#### INITIALIZE

In [ ]:
# Initialize the model
baggingModel = BaggingRegressor()

#### FIT/TRAIN

In [ ]:
# Fit the model using training data
baggingModel = baggingModel.fit(X_train, y_train)

#### PREDICT

In [ ]:
# Predictions from BaggingRegressor() using test data
y_test_pred = baggingModel.predict(X_test)

#### EVALUATE

In [ ]:
# Compute the Mean Squared Error using test data
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_test_pred)
print(f"Mean Squared Error: {mse}")

# <font color="pink">Stacking: Classification</font>

### Import Libraries

In [ ]:
# Importing pandas for data manipulation and analysis
import pandas as pd

# Importing numpy for numerical computations
import numpy as np

# Import the StackingClassifier for building stacked ensemble models
from sklearn.ensemble import StackingClassifier

# Import base models to be used in stacking
from sklearn.tree import DecisionTreeClassifier  # Decision tree classifier
from sklearn.neighbors import KNeighborsClassifier  # K-nearest neighbors classifier
from sklearn.naive_bayes import GaussianNB  # Naive Bayes classifier

# Import Logistic Regression for creating the meta model
from sklearn.linear_model import LogisticRegression

### Data Import and Pre-Processing

In [ ]:
# Load the CSV file 'flights.csv' into a pandas DataFrame and drop rows with missing values
# flights = pd.read_csv('flights.csv').dropna().sample(1000)
flights = pd.read_csv('flights.csv').dropna()
# Display the flights DataFrame
flights.head()

### Scikit-Learn Workflow

#### DEFINE

In [ ]:
# Create input features X and output feature y from the flights dataset
X = flights[[
    'temp',        # Temperature
    'dewp',        # Dew point
    'humid',       # Humidity
    'wind_dir',    # Wind direction
    'wind_speed',  # Wind speed
    'wind_gust',   # Wind gust speed
    'precip',      # Precipitation amount
    'pressure',    # Atmospheric pressure
    'visib'        # Visibility
]]
y = flights[['delay']]  # Target variable: delay

In [ ]:
from sklearn.model_selection import train_test_split

# Split X and y into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=50,
    stratify=y  # Ensure same ratio of delayed to non-delayed flights in both sets
)

# np.ravel(y): converts y from 2D array to 1D array as required by sklearn
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

#### INITIALIZE

In [ ]:
# Create a list of base models for stacking ensemble
estimators = [
    ('decision_tree', DecisionTreeClassifier()),  # Decision tree as first base model
    ('knn', KNeighborsClassifier()),              # KNN as second base model
    ('gnb', GaussianNB()),                        # Gaussian Naive Bayes as third base model
]

In [ ]:
# Initialize the stacking model with base models and a logistic regression final estimator
stackingModel = StackingClassifier(
    estimators=estimators,                   # List of base learners
    final_estimator=LogisticRegression()     # Meta-learner for final prediction
)

#### FIT/TRAIN

In [ ]:
# Fit the stacking model to the training data
stackingModel = stackingModel.fit(
    X_train,                                       # Input features
    y_train                              # Flattened target variable
)

#### PREDICT

In [ ]:
# Make predictions using the test data
Y_test_pred = stackingModel.predict(X_test)

#### EVALUATE

Create the Confusion Matrix

In [ ]:
# Generate the transfer matrix
pd.crosstab(
    y_test, # Actual values
    y_test_pred, # Predicted values
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
# Calculate and print the prediction accuracy of the stacked model on training data
stackingModel.score(
    X_test,   # Input features
    y_test    # True target values
)

# <font color="pink">Stacking: Regression</font>

### Import Libraries

In [ ]:
# Import pandas library
import pandas as pd
# Import numpy library for numerical computations and array operations
import numpy as np
# Import StackingRegressor for combining multiple regression models
from sklearn.ensemble import StackingRegressor
# Import LinearRegression model for simple linear regression
from sklearn.linear_model import LinearRegression
# Import Support Vector Regressor for non-linear regression
from sklearn.svm import SVR
# Import DecisionTreeRegressor for tree-based regression
from sklearn.tree import DecisionTreeRegressor

### Data Import and Pre-Processing

In [ ]:
flights = pd.read_csv('flights.csv').dropna()

flights.head()

### Scikit-Learn Workflow

#### DEFINE

In [ ]:
# Create input features X and output feature y
X = flights[
    [
        'dewp',
        'humid',
        'wind_dir',
        'wind_speed',
        'wind_gust',
        'precip',
        'pressure',
        'visib',
    ]
]
y = flights[['temp']]

In [ ]:
from sklearn.model_selection import train_test_split

# Split X and y into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=50
)

# np.ravel(y): converts y from 2D array to 1D array as required by sklearn
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

#### INITIALIZE

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

# Create a list of base models
estimators = [
# This line initializes a Python list named estimators. Each element in this list is a tuple representing a base model.
    ('linear', LinearRegression()),
    ('decision_tree', DecisionTreeRegressor()),
    ('svr', make_pipeline(StandardScaler(), SVR())),
    # This adds a Support Vector Regressor (SVR). It's wrapped in a make_pipeline
    # with a StandardScaler to scale the data before fitting the SVR.
    # This is often necessary for SVM models. It's named 'svr'.
    ('poly', make_pipeline(PolynomialFeatures(degree=2), LinearRegression())),
    # This adds a linear regression model that works on polynomial features
    # of degree 2. This is also wrapped in a make_pipeline and named 'poly'.
]

In [ ]:
# Initialize the model
stackingModel = StackingRegressor(
    estimators=estimators, final_estimator=LinearRegression()
)

#### FIT/TRAIN

In [ ]:
# Fit the model
stackingModel = stackingModel.fit(X_train, y_train)

#### PREDICT

In [ ]:
# Predict using test data
y_test_pred = stackingModel.predict(X_test)

#### EVALUATE

In [ ]:
# Computing the Mean Squared Error
from sklearn.metrics import mean_squared_error

# Compute the Mean Squared Error
mse = mean_squared_error(y_test, y_test_pred)
#

# Display the Mean Squared Error
print(f"Mean Squared Error: {mse}")